# Lesson 04: Training on COD10K

Lessons 01–03 built a model. Now we **train** it on all 3040 COD10K training images and **test** it on the 2026 test images.

**The training recipe** (the same for every model you will ever train):

```
data  ─► model ─► loss ─► backward ─► optimizer step      (repeat for every batch = 1 epoch)
                                                            after each epoch: evaluate on validation,
                                                            save the model if it's the best so far
at the very end: load the best model, evaluate ONCE on the test set
```

| Part | Topic |
|---|---|
| A | Data: train / validation / test split, OpenCV augmentation, `Dataset` and `DataLoader` |
| B | Model: pretrained ResNet50 encoder + UNet decoder (from Lessons 02–03) |
| C | Loss (BCE + IoU) and metrics (MAE, IoU, Dice) |
| D | The training loop, with checkpoints |
| E | Learning curves |
| F | Test-set results + the best and worst predictions |
| G | Your experiment log: compare runs as you modify the model |

> **Run it twice.** First with `QUICK = True` (a small subset, about 1 minute) to check that everything works.
> Then set `QUICK = False` and **Restart & Run All** for the real training (about 20–30 min on your RTX 4050).
>
> The QUICK run's scores will be poor (only 2 short epochs). It only checks that nothing crashes, and it adds a row to the results log marked `quick=True` that you can ignore.

## Settings

Everything you might want to change is in this one cell.

In [ ]:
RUN_NAME = "resnet50_unet_concat"   # ✏️ give every experiment its own name (used for the checkpoint + log)
QUICK = True                        # ✏️ True = small subset to check it works; False = real training

EPOCHS = 2 if QUICK else 25
BATCH_SIZE = 8                      # fits easily in 6 GB with AMP (Lesson 02, Part E)
LR = 1e-4                           # learning rate for Adam
NUM_WORKERS = 4                     # parallel image loading; set to 0 if you get DataLoader errors
VAL_FRACTION = 0.1                  # 10% of the training images are held out for validation
SEED = 42

PRETRAINED = True
DECODER_CFG = dict(decoder_channels=(256, 128, 64, 32), merge="concat", up="bilinear")

In [ ]:
import csv
import os
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from blocks import count_params, show
from config import IMAGE_SIZE, OUTPUT_DIR, cod10k_pairs, get_device
from data import CODDataset, augment
from decoders import SegModel, UNetDecoder
from encoders import ResNet50Encoder

device = get_device()
torch.manual_seed(SEED)
print(f"device: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

CHECKPOINT_DIR = os.path.join(os.path.dirname(OUTPUT_DIR), "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## Part A: data

### Three splits, three jobs

| split | images | used for |
|---|---|---|
| **train** | 90% of COD10K Train | learning the weights |
| **validation** | 10% of COD10K Train | choosing the best epoch, comparing settings |
| **test** | COD10K Test (2026) | the final number you report, **used once at the end** |

If you tuned your settings by looking at the test score, the test score would no longer be honest.
That's the same rule as in your diabetes notebooks: fit on train, choose on validation, report on test.

In [ ]:
all_train = cod10k_pairs("Train")
test_pairs = cod10k_pairs("Test")

rng = np.random.default_rng(SEED)
order = rng.permutation(len(all_train))
n_val = int(len(all_train) * VAL_FRACTION)
val_pairs = [all_train[i] for i in order[:n_val]]
train_pairs = [all_train[i] for i in order[n_val:]]

if QUICK:
    train_pairs, val_pairs, test_pairs = train_pairs[:160], val_pairs[:40], test_pairs[:100]

print(f"train {len(train_pairs)}   val {len(val_pairs)}   test {len(test_pairs)}" + ("   (QUICK subset)" if QUICK else ""))

### Data augmentation with OpenCV

3040 images is small for a deep network, and it can start to **memorise** them (overfitting).
Augmentation creates a slightly different version of each image every epoch, so the model has to learn the *object*, not the exact pixels.

The functions are in `data.py`, written with the OpenCV calls you already know:

| augmentation | OpenCV | image | mask |
|---|---|---|---|
| horizontal flip | `cv2.flip` | ✔ | ✔ same flip |
| random zoom-in crop | array slicing | ✔ | ✔ same crop |
| small rotation | `cv2.getRotationMatrix2D` + `cv2.warpAffine` | ✔ | ✔ same rotation (`INTER_NEAREST`) |
| colour jitter | `cv2.cvtColor` to HSV | ✔ | ✘ colour doesn't change the object's shape |

Below: the same image, augmented 6 times.

In [ ]:
viz_set = CODDataset(train_pairs, size=IMAGE_SIZE, train=False)
image, mask = viz_set.load(0)

tiles, titles = [cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE))], ["original"]
for k in range(6):
    aug_img, aug_mask = augment(image, mask, np.random.default_rng(k))
    aug_img = cv2.resize(aug_img, (IMAGE_SIZE, IMAGE_SIZE))
    aug_mask = cv2.resize(aug_mask, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_NEAREST)
    contours, _ = cv2.findContours((aug_mask > 127).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(aug_img, contours, -1, (0, 255, 0), 2)   # GT outline in green
    tiles.append(aug_img)
    titles.append(f"augmented {k + 1}")
show(tiles, titles, cols=4, size=3.5)

The green outline is the **augmented mask**. It must stay exactly on the animal. If it doesn't, the augmentation has a bug.

### `Dataset` and `DataLoader`

- `CODDataset[i]` loads one image + mask → augments (train only) → resizes to 352 → returns tensors.
- `DataLoader` groups them into **batches**, **shuffles** the training set every epoch, and loads images in parallel (`num_workers`).

In [ ]:
train_set = CODDataset(train_pairs, size=IMAGE_SIZE, train=True)
val_set = CODDataset(val_pairs, size=IMAGE_SIZE, train=False)
test_set = CODDataset(test_pairs, size=IMAGE_SIZE, train=False)

loader_args = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=device.type == "cuda",
                   persistent_workers=NUM_WORKERS > 0)
train_loader = DataLoader(train_set, shuffle=True, drop_last=True, **loader_args)
val_loader = DataLoader(val_set, shuffle=False, **loader_args)
test_loader = DataLoader(test_set, shuffle=False, **loader_args)

images, masks = next(iter(train_loader))
print(f"one batch -> images {tuple(images.shape)}  masks {tuple(masks.shape)}  mask values {masks.unique().tolist()}")
print(f"batches per epoch: {len(train_loader)}")

## Part B: the model

The pretrained ResNet50 encoder (Lesson 02) and the UNet decoder (Lesson 03), joined by `SegModel`. The decoder settings come from `DECODER_CFG` in the settings cell.

In [ ]:
def build_model():
    encoder = ResNet50Encoder(pretrained=PRETRAINED)
    decoder = UNetDecoder(encoder.channels, **DECODER_CFG)
    return SegModel(encoder, decoder)


model = build_model().to(device)
print(f"params: encoder {count_params(model.encoder):,}   decoder {count_params(model.decoder):,}   "
      f"total {count_params(model):,}")

## Part C: loss and metrics

**Loss = BCE + IoU loss.** In COD10K the object often covers less than 10% of the image.
With BCE alone, the model can get a low loss by predicting "background" nearly everywhere.
The IoU term measures overlap for **each image as a whole**, so a missed small object costs a lot.
(SINet-V2's *structure loss* is a weighted version of this same pair, covered in Lesson 07.)

**Metrics**
- **MAE** (mean absolute error): the standard COD metric. It's computed on the soft probability map, and **lower is better**.
- **IoU** and **Dice**: computed on the mask thresholded at 0.5. **Higher is better.**

In [ ]:
def bce_iou_loss(logits, target):
    """
    BCE  : judges every pixel on its own ("is this pixel object?")
    IoU  : judges the whole shape ("how much do prediction and GT overlap?")
    Together they work better than either alone, especially for small objects,
    where BCE is dominated by the (huge) background.
    """
    logits = logits.float()
    bce = F.binary_cross_entropy_with_logits(logits, target)
    prob = torch.sigmoid(logits)
    inter = (prob * target).sum(dim=(2, 3))
    union = (prob + target).sum(dim=(2, 3)) - inter
    iou_loss = 1 - (inter + 1) / (union + 1)      # +1 avoids 0/0 on empty masks
    return bce + iou_loss.mean()


@torch.no_grad()
def batch_metrics(logits, target, threshold=0.5):
    """
    Per-image metrics for a batch. Returns three lists (one value per image).
      MAE  : mean |prob - GT|, the standard COD metric (lower is better)
      IoU  : overlap of the thresholded mask with the GT (higher is better)
      Dice : 2*overlap / (pred + GT), also called F1 (higher is better)
    """
    prob = torch.sigmoid(logits.float())
    pred = (prob > threshold).float()
    mae = (prob - target).abs().mean(dim=(1, 2, 3))
    inter = (pred * target).sum(dim=(1, 2, 3))
    total = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    iou = torch.where(total - inter > 0, inter / (total - inter).clamp(min=1), torch.ones_like(inter))
    dice = torch.where(total > 0, 2 * inter / total.clamp(min=1), torch.ones_like(inter))
    return mae.tolist(), iou.tolist(), dice.tolist()

In [ ]:
# Quick check: a perfect prediction, a wrong one, and an "all background" one
gt = torch.zeros(1, 1, 64, 64)
gt[..., 20:40, 20:40] = 1
perfect = (gt * 20 - 10)          # large logits: +10 inside, -10 outside
inverted = -perfect
all_bg = torch.full_like(gt, -10.0)

for name, logits in [("perfect", perfect), ("inverted", inverted), ("all background", all_bg)]:
    m, i, d = batch_metrics(logits, gt)
    print(f"{name:<15} loss {bce_iou_loss(logits, gt).item():6.3f}   MAE {m[0]:.3f}   IoU {i[0]:.3f}   Dice {d[0]:.3f}")

Notice that "all background" has a **low MAE** (the object is only 10% of the image), but its IoU is 0.
That's why we always look at more than one metric.

## Part D: the training loop

`train_one_epoch` is the loop from Lesson 03 Part E, now over batches from the `DataLoader`.
`evaluate` runs the model on the validation set **without** augmentation and without gradients.

In [ ]:
def train_one_epoch(model, loader, optimizer, scaler, device, loss_fn=None):
    """One pass over the training data. Returns the average loss."""
    loss_fn = loss_fn or bce_iou_loss
    use_amp = device.type == "cuda"
    model.train()
    total, count = 0.0, 0
    for images, masks in tqdm(loader, desc="train", leave=False):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(images)
        loss = loss_fn(logits, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total += loss.item() * images.size(0)
        count += images.size(0)
    return total / count


@torch.no_grad()
def evaluate(model, loader, device, per_image=False):
    """Average MAE / IoU / Dice over a dataset (no augmentation, no gradients)."""
    use_amp = device.type == "cuda"
    model.eval()
    maes, ious, dices = [], [], []
    for images, masks in tqdm(loader, desc="eval", leave=False):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(images)
        m, i, d = batch_metrics(logits, masks)
        maes += m
        ious += i
        dices += d
    result = {"mae": float(np.mean(maes)), "iou": float(np.mean(ious)), "dice": float(np.mean(dices))}
    if per_image:
        result["per_image_iou"] = ious
    return result

### Train

- **Optimizer:** Adam, `LR = 1e-4`
- **Scheduler:** cosine. The learning rate slowly decreases to almost 0 by the last epoch.
- **Checkpoint:** after each epoch, if validation MAE is the best so far, save the weights to `checkpoints/<RUN_NAME>.pth`.

This cell runs for a while when `QUICK = False`. Progress bars show each epoch.

In [ ]:
model = build_model().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
checkpoint_path = os.path.join(CHECKPOINT_DIR, f"{RUN_NAME}.pth")

history = {"train_loss": [], "val_mae": [], "val_iou": [], "val_dice": []}
best_mae, best_epoch = float("inf"), -1
start = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    t0 = time.perf_counter()
    train_loss = train_one_epoch(model, train_loader, optimizer, scaler, device)
    val = evaluate(model, val_loader, device)
    lr = optimizer.param_groups[0]["lr"]
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_mae"].append(val["mae"])
    history["val_iou"].append(val["iou"])
    history["val_dice"].append(val["dice"])

    marker = ""
    if val["mae"] < best_mae:
        best_mae, best_epoch = val["mae"], epoch
        torch.save({"model": model.state_dict(), "epoch": epoch, "val": val,
                    "decoder_cfg": DECODER_CFG}, checkpoint_path)
        marker = "  ★ saved"
    print(f"epoch {epoch:>2}/{EPOCHS}  loss {train_loss:.4f}  |  val MAE {val['mae']:.4f}  IoU {val['iou']:.3f}  "
          f"Dice {val['dice']:.3f}  |  lr {lr:.1e}  {time.perf_counter() - t0:5.0f}s{marker}")

train_minutes = (time.perf_counter() - start) / 60
print(f"\nbest epoch {best_epoch}: val MAE {best_mae:.4f}   total {train_minutes:.1f} min   -> {checkpoint_path}")

## Part E: learning curves

How to read them:
- **Train loss keeps falling, but val MAE starts rising** → overfitting: the model memorises the training images. More augmentation, a smaller model or fewer epochs help.
- **Both still improving at the end** → train longer (more `EPOCHS`).
- **Nothing improves** → the learning rate is too low or too high, or there's a bug. (Did Lesson 03 Part E work?)

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(epochs, history["train_loss"], marker="o")
axes[0].set_title("train loss (BCE + IoU)")
axes[1].plot(epochs, history["val_mae"], marker="o", color="tab:red")
axes[1].axvline(best_epoch, color="gray", linestyle="--", label=f"best epoch {best_epoch}")
axes[1].set_title("val MAE (lower is better)")
axes[1].legend()
axes[2].plot(epochs, history["val_iou"], marker="o", label="IoU")
axes[2].plot(epochs, history["val_dice"], marker="o", label="Dice")
axes[2].set_title("val IoU / Dice (higher is better)")
axes[2].legend()
for ax in axes:
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Part F: test-set results

Load the **best** checkpoint (not the last epoch) and evaluate **once** on the test set.

> These numbers are computed at 352×352. Published COD papers resize each prediction back to the **original** image size before scoring, and also report S-measure and E-measure.
> Lesson 07 adds that full evaluation, so the numbers here are for **comparing your own runs**, not for comparing with papers yet.

In [ ]:
ckpt = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(ckpt["model"])
print(f"loaded best checkpoint from epoch {ckpt['epoch']}")

test = evaluate(model, test_loader, device, per_image=True)
print(f"TEST ({len(test_set)} images):  MAE {test['mae']:.4f}   IoU {test['iou']:.3f}   Dice {test['dice']:.3f}")

### Best and worst predictions

For each image: **green** = ground-truth outline, **red** = predicted outline (probability > 0.5).
Looking at the failures tells you what to improve next, and that's how new architecture ideas start.

In [ ]:
@torch.no_grad()
def predict(index):
    image_t, mask_t = test_set[index]
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device.type == "cuda"):
        logits = model(image_t[None].to(device))
    prob = torch.sigmoid(logits.float())[0, 0].cpu().numpy()
    image, _ = test_set.load(index)
    image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE))
    return image, (mask_t[0].numpy() > 0.5), prob


def outline(image, gt, prob):
    out = image.copy()
    for m, color in [(gt, (0, 255, 0)), (prob > 0.5, (0, 0, 255))]:
        contours, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(out, contours, -1, color, 2)
    return out


ious = np.array(test["per_image_iou"])
ranked = np.argsort(ious)
for label, idxs in [("BEST", ranked[::-1][:4]), ("WORST", ranked[:4])]:
    tiles, titles = [], []
    for i in idxs:
        image, gt, prob = predict(int(i))
        tiles += [outline(image, gt, prob), prob]
        name = os.path.basename(test_pairs[i][0]).replace("COD10K-CAM-", "")[:28]
        titles += [f"IoU {ious[i]:.2f}  {name}", "prediction"]
    print(label)
    show(tiles, titles, cmap="gray", cols=4, size=3.5)

## Part G: your experiment log

Every run adds one row to `outputs/results.csv`. When you change something (the decoder, the loss, the augmentation, the encoder ...),
give it a new `RUN_NAME`, run the notebook again, and compare the rows. This is how you'll test your own COD ideas later.

In [ ]:
log_path = os.path.join(OUTPUT_DIR, "results.csv")
row = {
    "run": RUN_NAME, "quick": QUICK, "epochs": EPOCHS, "best_epoch": best_epoch, "batch": BATCH_SIZE, "lr": LR,
    "pretrained": PRETRAINED, "decoder": str(DECODER_CFG), "params_M": round(count_params(model) / 1e6, 2),
    "val_mae": round(best_mae, 4), "test_mae": round(test["mae"], 4), "test_iou": round(test["iou"], 4),
    "test_dice": round(test["dice"], 4), "train_min": round(train_minutes, 1),
}
new_file = not os.path.exists(log_path)
with open(log_path, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=row.keys())
    if new_file:
        writer.writeheader()
    writer.writerow(row)

with open(log_path) as f:
    print(f.read())

> ✏️ **TRY IT**: one change per run, with a new `RUN_NAME` each time:
> 1. `DECODER_CFG = dict(decoder_channels=(256, 128, 64, 32), merge="add")`: concat vs add, on real data now.
> 2. `use_skips=(False, False, False, False)`: how much do skips matter for the **test** IoU? (Compare with Lesson 03 Part B.)
> 3. `PRETRAINED = False`: how much does ImageNet pretraining help?
> 4. In `data.py`, make `augment()` return the image and mask unchanged: how does the gap between train loss and val MAE change?
> 5. `BATCH_SIZE = 16` and `LR = 2e-4`: does it fit your GPU? Is it faster?
>
> Send me your `results.csv` after a few runs, and we'll read it together.

---
## Summary

- **Train / validation / test**: learn on train, choose on validation, report test **once**.
- **Augmentation** (flip, crop, rotate, colour) is done with plain OpenCV, and the image and mask must get the same geometric change.
- **BCE + IoU loss** handles the small-object imbalance of COD. Report **MAE** plus an overlap metric.
- **Checkpoint the best validation epoch**, not the last one.
- Keep an **experiment log**. Every architecture change you make from now on is a new row.

**Next lesson (05): UNet++**: nested skip connections, trained with this same recipe, and compared in your `results.csv`.